In [ ]:
import pandas as pd
from pathlib import Path

# =============================================================================
# INPUT / OUTPUT
# =============================================================================

INPUT_CSV = Path("data") / "MASTER_VARIABLES.csv"

DISTRICT_OUTPUT = Path("data") / "government_response_district.csv"
RC_OUTPUT = Path("data") / "government_response_rc.csv"

# =============================================================================
# LOAD DATA
# =============================================================================

df = pd.read_csv(INPUT_CSV)

print("Input shape:", df.shape)

# =============================================================================
# CLEAN KEYS
# =============================================================================

df["dtname"] = df["dtname"].astype(str).str.strip()
df["timeperiod"] = df["timeperiod"].astype(str).str.strip()

# =============================================================================
# PARAMETERS
# =============================================================================

FISCAL_YEAR_START_MONTH = 4

# =============================================================================
# Z-SCORE FUNCTION
# =============================================================================

def zscore(x):

    std = x.std(ddof=0)

    if std == 0:
        return pd.Series(0, index=x.index)

    return (x - x.mean()) / std

# =============================================================================
# GOVERNMENT RESPONSE CLASSIFICATION
# HIGHER SPENDING = LOWER RISK
# =============================================================================

def classify(z):

    if z <= -1.5:
        return 5

    elif z <= -0.5:
        return 4

    elif z <= 0.5:
        return 3

    elif z <= 1.5:
        return 2

    else:
        return 1

# =============================================================================
# FINANCIAL YEAR FUNCTION
# =============================================================================

def get_financial_year(tp):

    year, month = map(int, str(tp).split("_"))

    if month >= FISCAL_YEAR_START_MONTH:
        return f"{year}-{year + 1}"

    return f"{year - 1}-{year}"

# =============================================================================
# DISTRICT-MONTH TENDER TOTAL
# =============================================================================

district_df = (
    df.groupby(
        ["dtname", "timeperiod"],
        as_index=False
    )
    .agg(
        district_tender_value=(
            "total-tender-awarded-value",
            "sum"
        )
    )
)

# =============================================================================
# FINANCIAL YEAR + DATE
# =============================================================================

district_df["financial_year"] = (
    district_df["timeperiod"]
    .apply(get_financial_year)
)

district_df["date"] = pd.to_datetime(
    district_df["timeperiod"],
    format="%Y_%m"
)

district_df = district_df.sort_values(
    ["dtname", "date"]
)

# =============================================================================
# CUMULATIVE TENDER VALUE WITHIN FY
# =============================================================================

district_df["cum_tender_value"] = (
    district_df
    .groupby(
        ["dtname", "financial_year"]
    )["district_tender_value"]
    .cumsum()
)

# =============================================================================
# MONTHWISE STANDARDIZATION
# =============================================================================

district_df["govtresponse_z"] = (
    district_df
    .groupby("timeperiod")["cum_tender_value"]
    .transform(zscore)
)

# =============================================================================
# GOVERNMENT RESPONSE SCORE
# =============================================================================

district_df["government_response"] = (
    district_df["govtresponse_z"]
    .apply(classify)
)

# =============================================================================
# SAVE DISTRICT OUTPUT
# =============================================================================

district_df.to_csv(
    DISTRICT_OUTPUT,
    index=False
)

# =============================================================================
# MERGE DISTRICT SCORE BACK TO RCs
# =============================================================================

rc_df = df.copy()

if "government_response" in rc_df.columns:
    rc_df = rc_df.drop(
        columns=["government_response"]
    )

rc_df = rc_df.merge(
    district_df[
        [
            "dtname",
            "timeperiod",
            "district_tender_value",
            "cum_tender_value",
            "govtresponse_z",
            "government_response",
        ]
    ],
    on=["dtname", "timeperiod"],
    how="left",
    validate="many_to_one",
)

# =============================================================================
# SAVE RC OUTPUT
# =============================================================================

rc_df.to_csv(
    RC_OUTPUT,
    index=False
)

# =============================================================================
# SUMMARY
# =============================================================================

print(f"\nDistrict output : {DISTRICT_OUTPUT}")
print(f"RC output       : {RC_OUTPUT}")

print("\nDistrict rows:", len(district_df))
print("RC rows:", len(rc_df))

print("\nGovernment Response Distribution")

print(
    district_df["government_response"]
    .value_counts()
    .sort_index()
)

print("\nMissing values:")

print(
    rc_df["government_response"]
    .isna()
    .sum()
)

print("\nDistrict preview:")

print(
    district_df[
        [
            "dtname",
            "timeperiod",
            "district_tender_value",
            "cum_tender_value",
            "govtresponse_z",
            "government_response",
        ]
    ].head()
)

print("\nRC preview:")

print(
    rc_df[
        [
            "object_id",
            "revenue_circle",
            "dtname",
            "timeperiod",
            "government_response",
        ]
    ].head()
)